# AIC keyframe shard extraction

Duplicate this notebook once per shard and change only `SHARD` in the next cell. Use a CPU session and enable Internet so the pinned public repository tag can be cloned.

In [ ]:
from pathlib import Path
import os

SHARD = "L21/part-001.json"  # Change only this value per notebook.
WORKERS = min(4, os.cpu_count() or 1)
REPO_URL = "https://github.com/trungdangtapcode/FRED-Video-Search-System-Resumable.git"
REPO_REF = "kaggle-shards-v1"
DATASET_ROOTS = [
    Path("/kaggle/input/datasets/khoahunhtngng/aic2024-round1-data"),
    Path("/kaggle/input/aic2024-round1-data"),
]
print({"shard": SHARD, "workers": WORKERS})

In [ ]:
import shutil
import subprocess
import sys

repo_dir = Path("/kaggle/tmp/FRED-Video-Search-System-Resumable")
if repo_dir.exists():
    shutil.rmtree(repo_dir)
subprocess.run(
    ["git", "clone", "--quiet", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(repo_dir)],
    check=True,
)
try:
    import cv2
except ModuleNotFoundError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"],
        check=True,
    )
    import cv2
commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
).strip()
print({"repo": str(repo_dir), "commit": commit, "opencv": cv2.__version__})

In [ ]:
VIDEO_EXTENSIONS = {".avi", ".mkv", ".mov", ".mp4", ".webm"}

dataset_root = next((path for path in DATASET_ROOTS if path.is_dir()), None)
if dataset_root is None:
    raise FileNotFoundError(f"Dataset is not attached. Checked: {DATASET_ROOTS}")
candidates = [dataset_root / "vidoe", dataset_root / "video", dataset_root]
video_dir = None
for candidate in candidates:
    if candidate.is_dir() and any(
        path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
        for path in candidate.rglob("*")
    ):
        video_dir = candidate
        break
if video_dir is None:
    raise FileNotFoundError(f"No source videos found below {dataset_root}")

manifest_rel = Path(SHARD)
manifest_path = repo_dir / "manifests" / "aic25-b1" / manifest_rel
if not manifest_path.is_file():
    raise FileNotFoundError(f"Unknown shard manifest: {manifest_path}")
run_dir = Path("/kaggle/working/keyframes") / manifest_rel.parent / manifest_rel.stem
disk = shutil.disk_usage("/kaggle/working")
print({
    "dataset_root": str(dataset_root),
    "video_dir": str(video_dir),
    "manifest": str(manifest_path),
    "run_dir": str(run_dir),
    "working_free_gb": round(disk.free / 1e9, 2),
})

In [ ]:
command = [
    sys.executable,
    "-m",
    "keyframe_extraction",
    "--input",
    str(video_dir),
    "--manifest",
    str(manifest_path),
    "--run-dir",
    str(run_dir),
    "--workers",
    str(WORKERS),
]
print("Running:", " ".join(command))
subprocess.run(
    command,
    cwd=repo_dir,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
    check=True,
)

In [ ]:
import json

success_path = run_dir / "_SUCCESS.json"
if not success_path.is_file():
    raise RuntimeError(f"Shard did not produce its success marker: {success_path}")
success = json.loads(success_path.read_text(encoding="utf-8"))
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
expected_videos = len(manifest["videos"])
done_files = list((run_dir / "extraction_status").glob("*.done.json"))
png_files = list((run_dir / "keyframes").glob("*/*.png"))
if success["selected"] != expected_videos or len(done_files) != expected_videos:
    raise RuntimeError(
        f"Incomplete shard: expected={expected_videos}, "
        f"selected={success['selected']}, done={len(done_files)}"
    )
output_bytes = sum(path.stat().st_size for path in run_dir.rglob("*") if path.is_file())
print({
    "status": "complete",
    "shard": manifest["shard_id"],
    "videos": expected_videos,
    "frames": len(png_files),
    "output_gb": round(output_bytes / 1e9, 2),
    "success_marker": str(success_path),
})

After the final cell reports `status: complete`, use **Save Version -> Save & Run All** with output saving enabled. Attach that notebook's output to downstream notebooks through **Add Input -> Notebook Output Files**.